# laser-ai training notebook

**Fresh training:** upload `bundle.zip` and run all cells.

**Resume sequencer-only (skip VAE):** also upload an existing `model.pt` (or `vae_only.pt`). The notebook will detect it and skip VAE training, training the sequencer for `seq_epochs=200` on standardized latents.

**End-to-end fine-tuning:** set `E2E_EPOCHS > 0` in cell 4. After MSE training, the sequencer is fine-tuned with chamfer/rgb loss on decoded VAE frames — forces predictions onto the manifold the decoder can actually use.

In [ ]:
# 1. Install laser-ai from GitHub (force-reinstall to bypass pip cache)
!pip uninstall -y laser-ai
!pip install -q --upgrade pip
!pip install -q --no-cache-dir git+https://github.com/tobytorgerson-art/laser-ai.git

In [ ]:
# 2. Detect uploads
import pathlib
zips = list(pathlib.Path('.').glob('*.zip'))
assert zips, 'Upload bundle.zip to the file panel first'
BUNDLE = str(zips[0])

RESUME = None
for name in ('model.pt', 'vae_only.pt'):
    if pathlib.Path(name).exists():
        RESUME = name
        break

print('bundle:', BUNDLE)
print('resume:', RESUME if RESUME else '(none — full fresh run)')

In [ ]:
# 3. Sanity check: confirm installed code has the standardization + e2e fixes
import importlib, inspect, laser_ai.training.train_sequencer as ts
importlib.reload(ts)
src = inspect.getsource(ts)
assert 'latent_mean' in src and 'lats_norm' in src, 'Stale install — restart runtime and rerun cell 1'
assert 'train_sequencer_e2e' in src, 'E2E code missing — restart runtime and rerun cell 1'
print('OK: standardization + e2e code present')

In [ ]:
# 4. Run training
from laser_ai.colab_train import run

# Knobs you might want to tweak:
SEQ_EPOCHS = 200    # MSE pre-training epochs (resume) or default if fresh
E2E_EPOCHS = 50     # End-to-end fine-tuning epochs. Set 0 to skip.

OUT = 'model_new.pt' if RESUME else 'model.pt'
if RESUME:
    run(BUNDLE, OUT,
        resume_from_vae=RESUME,
        seq_epochs=SEQ_EPOCHS,
        e2e_epochs=E2E_EPOCHS)
else:
    run(BUNDLE, OUT, e2e_epochs=E2E_EPOCHS)
print('\nTraining complete. Download', OUT, 'from the file panel.')